# NB6 · Whole-body allocation and supervision

Use the spatial extension of the canonical four-bar with four joint moments.
Compare weighted QP and hierarchical QP on one compliant simulation plant.
This is a thin-disk research model for horizontal support surfaces with synthetic transverse inertia,
contact, driver bounds and sensor channels. The allocator uses individual normal
and longitudinal constraints plus a common lateral rolling row. The plant
retains separate per-wheel forces and allows slip and separation.

[DUK v3 chapter 08](https://app.notion.com/p/4131f3cc03328156af1dc6b76714fce1)
and [chapter 09](https://app.notion.com/p/4131f3cc0332811da657f1bbb15cb66b).

In [ ]:
import hashlib
import json
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from duk_core.modeling import load_leg_geometry_json

repo = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "robot/config/physical/duk_leg_geometry.json").is_file()
)
geometry_path = repo / "robot/config/physical/duk_leg_geometry.json"
fixture_path = repo / "research/experiments/duk-v3/synthetic.json"
geometry = load_leg_geometry_json(geometry_path)
fixture = json.loads(fixture_path.read_text())
assert fixture["schema_version"] == 1 and fixture["status"] == "synthetic_research_only"
print(
    "Working tree dirty:",
    bool(subprocess.check_output(["git", "status", "--porcelain"], cwd=repo)),
)
print(
    "Commit",
    subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip(),
)
for path in (geometry_path, fixture_path):
    print(path.relative_to(repo), hashlib.sha256(path.read_bytes()).hexdigest())
print(fixture["source"])

In [ ]:
from collections import Counter

from duk_core.modeling import build_planar_biped_model
from duk_core.modeling.hip_load import LinearTorqueSpeed
from duk_core.modeling.planar_numeric import NumericPlanarBiped
from duk_core.modeling.spatial_biped import SpatialBiped
from duk_core.simulation.whole_body import run_whole_body

planar = NumericPlanarBiped(build_planar_biped_model(geometry), fixture["planar"])
s = fixture["spatial"]
plant = SpatialBiped(planar, s["lateral_mounts_m"], s["transverse_inertias_kg_m2"])
p = fixture["whole_body"]
env = fixture["hip_envelope"]
motor = LinearTorqueSpeed(
    env["stall_torque_N_m"], env["no_load_speed_rad_s"], env["efficiency"]
)
q = np.r_[0.08, 0.45, 0.13, np.deg2rad([94.0, 103.0]), 0.12, -0.22]
v = np.array([0.12, -0.04, 0.2, 0.3, -0.2, -0.7, 0.6])
state, tangent = plant.sagittal_state(q, v)
d = plant.evaluate(state)
flat = planar.evaluate(q, v)
np.testing.assert_allclose(tangent.T @ d.mass @ tangent, flat.mass, atol=1e-10)
np.testing.assert_allclose(tangent.T @ d.actuation, flat.input, atol=1e-10)
np.testing.assert_allclose(
    d.kinematics.contact[:, 0] @ tangent, flat.contact[2:], atol=1e-10
)
np.testing.assert_allclose(
    d.kinematics.contact[:, 2] @ tangent, flat.contact[:2], atol=1e-10
)
assert np.linalg.eigvalsh(d.mass)[0] > 0.0
print("Spatial mass, actuation and contact agree in the sagittal reduction.")


def run(name, method="weighted", priority="balance_first"):
    scenario = p["scenario"] | p["cases"][name]
    candidate = plant
    if "support_levels_m" in scenario:
        candidate = SpatialBiped(
            planar,
            s["lateral_mounts_m"],
            s["transverse_inertias_kg_m2"],
            support_levels_m=scenario["support_levels_m"],
        )
    return run_whole_body(
        candidate,
        motor,
        fixture["v2_balance"],
        p,
        scenario,
        method=method,
        priority=priority,
    )


results = {name: run(name) for name in p["cases"]}
results["competing_hqp_balance"] = run("competing", "hierarchical", "balance_first")
results["competing_hqp_legs"] = run("competing", "hierarchical", "legs_first")
for name, trace in results.items():
    assert np.all(abs(trace.applied_effort) <= trace.effort_limit + 1e-8)
    assert np.all(trace.force[:, :, 2] >= 0.0)
    solved = np.array(trace.solver_status) == "solved"
    assert np.all(trace.solver_residual[solved] <= 2e-6)
    assert np.isfinite(trace.pose).all()
    peak = np.rad2deg(trace.peak_abs_lean_roll)
    reached_horizon = trace.stop_reason is None and np.isclose(
        trace.terminal_time, (p["scenario"] | p["cases"].get(name, {}))["duration_s"]
    )
    performance = (
        reached_horizon
        and peak[0] <= p["maximum_lean_deg"]
        and peak[1] <= p["maximum_roll_deg"]
    )
    print(
        name,
        "completed with declared angles:",
        performance,
        "peak lean/roll (deg):",
        peak.round(3),
        "solver states:",
        dict(Counter(trace.solver_status)),
        "stop:",
        trace.stop_reason,
        "max solve ms:",
        round(1000 * trace.solver_seconds.max(), 2),
    )

In [ ]:
nominal = results["nominal"]
assert len(nominal.time) == round(p["scenario"]["duration_s"] / p["period_s"])
assert nominal.terminal_time == p["scenario"]["duration_s"]
assert nominal.stop_reason is None and all(
    status == "solved" for status in nominal.solver_status
)
assert np.rad2deg(nominal.peak_abs_lean_roll[0]) <= p["maximum_lean_deg"]
assert np.rad2deg(nominal.peak_abs_lean_roll[1]) <= p["maximum_roll_deg"]
assert np.max(abs(nominal.pose[:, 3])) > 0.005
assert "Active" in [status[0] for status in nominal.feature_status]

off = results["drive_release"]
assert not off.feature_intent[:, 1:].any()
release = off.time >= p["cases"]["drive_release"]["drive_end_s"]
assert np.all(off.effective_reference[release, 2] == 0.0)

slip = results["low_grip"]
sliding = np.array(["sliding" in modes for modes in slip.contact_modes])
assert sliding.any() and np.all(slip.feature_amount[sliding] == 0.0)
assert "Paused" in [status[2] for status in slip.feature_status]

limited = results["competing_hqp_legs"]
bounce_states = [status[0] for status in limited.feature_status]
assert "Limited" in bounce_states and "Exit" in bounce_states
exit_index = bounce_states.index("Exit")
assert (
    0
    < limited.feature_amount[exit_index, 0]
    < limited.feature_amount[exit_index - 1, 0]
    < 1.0
)

late, infeasible = results["late"], results["infeasible"]
assert "deadline" in late.solver_status and "fallback" in late.supervisor
assert "infeasible" in infeasible.solver_status and "fallback" in infeasible.supervisor
brief = results["brief_late"]
assert "deadline" in brief.solver_status and brief.supervisor[-1] == "normal"
sensor = results["sensor_lost"]
assert sensor.stop_reason == "sensor" and np.all(sensor.applied_effort[-1] == 0.0)
assert (
    sensor.solver_status[-1] == "skipped_sensor" and sensor.command_source[-1] == "stop"
)
fallback = np.array(brief.command_source) == "balance_fallback"
assert fallback.any() and np.all(brief.requested_reference[fallback, 2] > 0.0)
assert np.all(brief.effective_reference[fallback, 1:] == 0.0)
assert np.all(brief.feature_amount[fallback] == 0.0)
air = results["extended_flight"]
assert ("flight", "flight") in air.contact_modes and air.stop_reason == "flight_or_fall"
brief_air = results["brief_flight"]
assert ("flight", "flight") in brief_air.contact_modes and brief_air.contact_modes[
    -1
] == ("rolling", "rolling")
one_wheel = results["one_wheel"]
assert any(modes.count("flight") == 1 for modes in one_wheel.contact_modes)
for key in ("brief_flight", "one_wheel"):
    event = results[key]
    print(key, "observed contact sequence:", tuple(dict.fromkeys(event.contact_modes)))
levels = results["split_level"]
assert abs(levels.pose[0, 2]) > 0.02
assert abs(levels.terminal_pose[2]) < abs(levels.pose[0, 2])
print(
    "Split-level roll initial/final (deg):",
    np.rad2deg([levels.pose[0, 2], levels.terminal_pose[2]]).round(3),
)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 6), constrained_layout=True)
axes[0, 0].plot(nominal.time, nominal.requested_reference[:, 0], label="Requested")
axes[0, 0].plot(nominal.time, nominal.effective_reference[:, 0], label="Admitted")
axes[0, 0].plot(nominal.time, nominal.pose[:, 1], label="Actual")
axes[0, 0].set_ylabel("Body height (m)")
axes[0, 0].legend()
axes[0, 1].plot(nominal.time, nominal.requested_reference[:, 1], label="Requested")
axes[0, 1].plot(nominal.time, nominal.effective_reference[:, 1], label="Admitted")
axes[0, 1].plot(nominal.time, nominal.pose_rate[:, 3], label="Actual")
axes[0, 1].set_ylabel("Yaw rate (rad/s)")
axes[0, 1].legend()
axes[0, 2].plot(slip.position_xy[:, 0], slip.position_xy[:, 1], label="Low grip")
axes[0, 2].plot(nominal.position_xy[:, 0], nominal.position_xy[:, 1], label="Nominal")
axes[0, 2].set(xlabel="x (m)", ylabel="y (m)")
axes[0, 2].legend()
for key, label in [
    ("competing", "Weighted"),
    ("competing_hqp_balance", "HQP balance first"),
    ("competing_hqp_legs", "HQP legs first"),
]:
    trace = results[key]
    axes[1, 0].plot(trace.time, np.rad2deg(trace.pose[:, 0]), label=label)
axes[1, 0].set_ylabel("Equivalent lean (deg)")
axes[1, 0].legend()
axes[1, 1].plot(slip.time, slip.force[:, :, 2], label=["Left", "Right"])
axes[1, 1].set_ylabel("Normal force (N)")
axes[1, 1].legend()
axes[1, 2].plot(
    slip.time, slip.feature_amount, label=["Bounce", "Level", "Anchor", "Heading"]
)
axes[1, 2].set_ylabel("Feature participation")
axes[1, 2].legend()
for ax in [*axes[1], *axes[0, :2]]:
    ax.set_xlabel("Time (s)")
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(10, 5), constrained_layout=True)
for i, axis in enumerate(axes.flat):
    axis.plot(late.time, late.requested_effort[:, i], "--", label="Candidate/request")
    axis.plot(late.time, late.applied_effort[:, i], label="Applied")
    axis.set(
        xlabel="Time (s)", ylabel=("Hip L", "Hip R", "Wheel L", "Wheel R")[i] + " (N·m)"
    )
    axis.legend()
plt.show()
fig, axes = plt.subplots(1, 2, figsize=(9, 3), constrained_layout=True)
axes[0].plot(levels.time, np.rad2deg(levels.pose[:, 2]), label="Body roll")
axes[0].axhline(0.0, linestyle="--", color="k", label="Level reference")
axes[0].set(xlabel="Time (s)", ylabel="Split-level roll (deg)")
axes[1].plot(levels.time, levels.force[:, :, 2], label=["Left", "Right"])
axes[1].set(xlabel="Time (s)", ylabel="Split-level normal force (N)")
for ax in axes:
    ax.legend()
plt.show()

**Scope of the comparison.** The supervisor consumes simulated pose and an
independently compensated acceleration channel. Contact observations and
applied forces come from the simulation. Low-grip and fault cases expose the
consequences of the selected research policy, including stopped trials. The
headroom governor uses measured command utilization from the preceding sample.
Hierarchy and solver times are evaluated on this Python process. Physical
sensor, drivetrain, road and stop-policy validation require separately
identified data.

Controller rows record the state before each held command. `terminal_pose`
records the final integrated state. Peak lean/roll includes every integration
substep, where fall and hip travel guards are checked. The QP bounds cover
the entire constant-acceleration hip prediction, including an interior turning
point. These remain checks of the numerical prediction and sampled plant.
Rolling odometry uses the tilted disk support geometry and compensated body,
carrier and wheel rates without using world translation.
